In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from src.data.priority_labeler import (
    assign_priority_labels,
    validate_priority_labels,
    PRIORITY_COLUMN
)
from src.models.trainer import (
    build_logistic_regression,
    build_random_forest,
    build_linear_svc,
    train_model,
    save_model
)
from src.models.evaluator import (
    evaluate_model,
    print_classification_report,
    plot_confusion_matrix,
    save_metrics_report,
    plot_model_comparison
)
from src.utils.config import (
    MODELS_DIR,
    PROCESSED_DATA_DIR,
    CATEGORY_TARGET,
    CATEGORY_TO_PRIORITY,
    PRIORITY_LABELS,
    TEXT_COLUMN
)

sns.set_theme(style="whitegrid")
print("Imports successful")

In [ ]:
train_clean = pd.read_csv(PROCESSED_DATA_DIR / "train_cleaned.csv")
test_clean  = pd.read_csv(PROCESSED_DATA_DIR / "test_cleaned.csv")

# Assign rule-based priority labels
train_priority = assign_priority_labels(train_clean)
test_priority  = assign_priority_labels(test_clean)

print("=== TRAIN PRIORITY DISTRIBUTION ===")
print(train_priority[PRIORITY_COLUMN].value_counts())

print("\n=== TEST PRIORITY DISTRIBUTION ===")
print(test_priority[PRIORITY_COLUMN].value_counts())

print("\n=== VALIDATION ===")
validate_priority_labels(train_priority)
validate_priority_labels(test_priority)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

priority_order = ["Low", "Medium", "High", "Critical"]
colors = {
    "Low":      "#2ca02c",
    "Medium":   "#1f77b4",
    "High":     "#ff7f0e",
    "Critical": "#d62728"
}

for ax, (df, title) in zip(axes, [
    (train_priority, "Training Set"),
    (test_priority,  "Test Set")
]):
    counts = df[PRIORITY_COLUMN].value_counts().reindex(priority_order)
    palette = [colors[p] for p in priority_order]

    sns.barplot(x=counts.index, y=counts.values, ax=ax, palette=palette)
    ax.set_title(f"Priority Distribution - {title}", fontweight="bold")
    ax.set_xlabel("Priority")
    ax.set_ylabel("Count")

    for i, val in enumerate(counts.values):
        ax.text(i, val + 3, str(val), ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("../outputs/figures/priority_distribution.png", dpi=150)
plt.show()

In [ ]:
print("=== CATEGORY TO PRIORITY MAPPING ===\n")
print(f"{'Category':<25} {'Priority':<12} {'Train Count'}")
print("-" * 50)

for category, priority in CATEGORY_TO_PRIORITY.items():
    count = train_priority[
        train_priority[CATEGORY_TARGET] == category
    ].shape[0]
    print(f"{category:<25} {priority:<12} {count}")

In [ ]:
splits = joblib.load(MODELS_DIR / "train_test_splits.pkl")

X_train_tfidf = splits["X_train_tfidf"]
X_test_tfidf  = splits["X_test_tfidf"]

# Get priority labels aligned to the same indices as our splits
y_pri_train = train_priority[PRIORITY_COLUMN].reset_index(drop=True)
y_pri_test  = test_priority[PRIORITY_COLUMN].reset_index(drop=True)

print(f"Training matrix: {X_train_tfidf.shape}")
print(f"Test matrix:     {X_test_tfidf.shape}")
print(f"\nTrain priority counts:")
print(y_pri_train.value_counts())

In [ ]:
lr_model = build_logistic_regression()
lr_model = train_model(lr_model, X_train_tfidf, y_pri_train)

In [ ]:
lr_metrics, lr_preds = evaluate_model(
    lr_model, X_test_tfidf, y_pri_test, "Logistic Regression"
)

print_classification_report(y_pri_test, lr_preds, "Logistic Regression")

plot_confusion_matrix(
    y_pri_test, lr_preds,
    "Logistic Regression",
    PRIORITY_LABELS,
    "priority_confusion_matrix_lr.png"
)

In [ ]:
rf_model = build_random_forest()
rf_model = train_model(rf_model, X_train_tfidf, y_pri_train)

In [ ]:
rf_metrics, rf_preds = evaluate_model(
    rf_model, X_test_tfidf, y_pri_test, "Random Forest"
)

print_classification_report(y_pri_test, rf_preds, "Random Forest")

plot_confusion_matrix(
    y_pri_test, rf_preds,
    "Random Forest",
    PRIORITY_LABELS,
    "priority_confusion_matrix_rf.png"
)

In [ ]:
svc_model = build_linear_svc()
svc_model = train_model(svc_model, X_train_tfidf, y_pri_train)

In [ ]:
svc_metrics, svc_preds = evaluate_model(
    svc_model, X_test_tfidf, y_pri_test, "LinearSVC"
)

print_classification_report(y_pri_test, svc_preds, "LinearSVC")

plot_confusion_matrix(
    y_pri_test, svc_preds,
    "LinearSVC",
    PRIORITY_LABELS,
    "priority_confusion_matrix_svc.png"
)

In [ ]:
all_metrics = [lr_metrics, rf_metrics, svc_metrics]

plot_model_comparison(all_metrics, "priority_model_comparison.png")

print("\n=== FINAL COMPARISON TABLE ===")
df_results = pd.DataFrame(all_metrics)
df_results = df_results.sort_values("f1_score", ascending=False)
print(df_results.to_string(index=False))

In [ ]:
best = max(all_metrics, key=lambda x: x["f1_score"])
print(f"Best model: {best['model']} with F1={best['f1_score']}")

model_map = {
    "Logistic Regression": lr_model,
    "Random Forest":       rf_model,
    "LinearSVC":           svc_model,
}

best_model = model_map[best["model"]]
save_model(best_model, "priority_classifier.pkl")
save_metrics_report(all_metrics, "priority_metrics.json")

print(f"\nBest priority model saved as priority_classifier.pkl")

In [ ]:
X_test_raw = splits["X_test"]

error_df = pd.DataFrame({
    "text":      X_test_raw.values,
    "true":      y_pri_test.values,
    "predicted": model_map[best["model"]].predict(X_test_tfidf)
})

errors = error_df[error_df["true"] != error_df["predicted"]]

print(f"Total errors: {len(errors)} out of {len(error_df)} ({len(errors)/len(error_df)*100:.1f}%)")

print(f"\n=== MOST COMMON PRIORITY CONFUSIONS ===")
confusion_pairs = errors.groupby(
    ["true", "predicted"]
).size().sort_values(ascending=False)
print(confusion_pairs.head(10))

print(f"\n=== SAMPLE MISCLASSIFIED TICKETS ===")
for _, row in errors.head(5).iterrows():
    print(f"\nTrue Priority:      {row['true']}")
    print(f"Predicted Priority: {row['predicted']}")
    print(f"Text:               {str(row['text'])[:200]}")

In [ ]:
# Show what the full system produces for sample tickets
cat_model = joblib.load(MODELS_DIR / "category_classifier.pkl")
pri_model = best_model
vectorizer = joblib.load(MODELS_DIR / "tfidf_vectorizer.pkl")

sample_tickets = [
    "Cannot access shared folder on network drive",
    "Need to reset password for new employee account",
    "Outlook keeps crashing when opening attachments",
    "Request to decommission old server hardware",
    "Printer not connecting to workstation",
]

print("=== END TO END PREDICTIONS ===\n")
print(f"{'Ticket':<50} {'Category':<25} {'Priority'}")
print("-" * 90)

from src.data.preprocessor import clean_text

for ticket in sample_tickets:
    cleaned   = clean_text(ticket)
    features  = vectorizer.transform([cleaned])
    category  = cat_model.predict(features)[0]
    priority  = pri_model.predict(features)[0]
    print(f"{ticket:<50} {category:<25} {priority}")